<div style="border: 5px solid black; padding: 20px; border-radius: 6px;">

## **Course:** DSC670 - Advanced Uses of Generative AI
## **Name:** Tim Hollis
## **Assignment:** Assessment 2.2 Problem 4
## **Date:** June 15, 2026

---

**Reference**

Bellevue University. (2024). *DSC 670 exercise invoice* [PDF]. https://content.bellevue.edu/cst/dsc/670/dsc-670-exercise-invoice.pdf

</div>

### **Initial Setup**

In [1]:
# Loading libraries
import os
import json
from getpass import getpass
from openai import OpenAI

# Prompting for the key at runtime with a masked input
api_key = getpass('Enter your OpenAI API key: ')
client = OpenAI(api_key=api_key)

# Global configuration
PDF_PATH = 'Invoice.pdf'
EXTRACTION_MODEL = 'gpt-4o'
JSON_MODEL = 'gpt-4o'

print('📦 Libraries loaded and OpenAI client ready')
print(f'📄 Target document: {PDF_PATH}')
print(f'🤖 Using model: {EXTRACTION_MODEL}')

Enter your OpenAI API key:  ········


📦 Libraries loaded and OpenAI client ready
📄 Target document: Invoice.pdf
🤖 Using model: gpt-4o


<div style="page-break-before: always;"></div>

## Step 1: Extract the Invoice Text with the OpenAI API

The assignment calls for the OpenAI API to do the extraction, so instead of parsing the PDF with a local library I hand the file directly to a vision-capable model and ask it to transcribe what it sees. I deliberately split the task into two API calls. This first one only reads the document and returns raw text; a later call handles the conversion to JSON. Keeping transcription separate from structuring lets me judge each step on its own, which matters because misreading the document and mis-organizing it are very different failures, and I want to know which one happened if something looks off.

In [2]:
# Upload the invoice to the OpenAI Files endpoint
uploaded = client.files.create(
    file=open(PDF_PATH, 'rb'),
    purpose='user_data'
)
print(f'📤 Uploaded {PDF_PATH} (file id: {uploaded.id})')

# Ask a vision-capable model to read the document and return its raw text
extraction_prompt = (
    'You are reading a business invoice. '
    'Transcribe every piece of text exactly as it appears, including labels, '
    'line items, dollar amounts, percentages, and footer notes. '
    'Preserve the values verbatim and do not summarize, reformat, or fill in blanks.'
)

extraction_response = client.responses.create(
    model=EXTRACTION_MODEL,
    input=[
        {
            'role': 'user',
            'content': [
                {'type': 'input_file', 'file_id': uploaded.id},
                {'type': 'input_text', 'text': extraction_prompt}
            ]
        }
    ]
)

invoice_text = extraction_response.output_text
print('📝 Extracted invoice text:\n')
print(invoice_text)

📤 Uploaded Invoice.pdf (file id: file-RGgNNig2uWbnrmKvosZn2z)
📝 Extracted invoice text:

<PARSED TEXT FOR PAGE: 1 / 1>

321 Avenue A Date: 6/28/2024  
Portland, OR 12345 Invoice # 1111  
Phone: (206) 555-1163 For PO # 123456  
Fax: (206) 555-1164  
someone@websitegoeshere.com  
Quantity Description Unit price Amount Discount applied  
1 Item Number 1 $ 2.00 2.00 $  
1 Item Number 2 $ 2.00 2.00 $  
1 Item Number 3 $ 2.00 2.00 $  
$ -  
$ -  
$ -  
$ -  
$ -  
$ -  
$ -  
$ -  
Subtotal 6.00 $  
Credit $ 1,000.00  
Tax 9.80%  
Additional discount 12%  
Balance due (994.20) $  
Company Name  
INVOICE  
If you have any questions concerning this invoice, contact  
<Name> at <phone or email>.  
Make all checks payable to <Company name>.  
Thank you for your business!  
Bill To:  
Natasha Jones  
Central Beauty  
123 Main St.  
% discount 10%  
Manhattan, NY 98765  
(321) 555-1234  
Items over this amount qualify for an additional discount $100


<div style="page-break-before: always;"></div>

## Step 2: Convert the Extracted Text to Structured JSON

The raw transcription is complete but disordered, so this second call asks the model to reorganize it into a clean JSON object with named sections. I gave it specific instructions for the judgment calls this invoice forces: keep the placeholder fields exactly as written instead of guessing, drop the empty dash-only rows so they do not pollute the line items, and represent the balance due as a negative number since the document presents it in parentheses. I am curious whether the model honors all three, because each one is a small test of whether it reorganizes faithfully or quietly "tidies up" the data in ways I did not request.

In [3]:
# Convert the extracted text into structured JSON with a second API call
json_prompt = (
    'Convert the following invoice text into a single structured JSON object. '
    'Use these top-level keys: seller, invoice_metadata, bill_to, line_items, totals, and notes. '
    'List each real line item as an object with quantity, description, unit_price, and amount, '
    'and ignore the empty placeholder rows that only contain a dash. '
    'Format every monetary amount as a string with a dollar sign and two decimal places, '
    'exactly as it appears on the invoice (for example, "$2.00" and "$1,000.00"). '
    'Show the balance due as a negative currency string because the invoice presents it in '
    'parentheses (for example, "-$994.20"). '
    'Keep placeholder values such as <Name> or <Company name> exactly as written rather than guessing. '
    'Capture percentage fields like tax and the discounts as their literal values. '
    'Return only valid JSON with no surrounding text or code fences.\n\n'
    f'Invoice text:\n{invoice_text}'
)

json_response = client.responses.create(
    model=JSON_MODEL,
    input=json_prompt
)

# Removing code fences if the model wrapped the JSON in them, then parsing
raw_json = json_response.output_text.strip()
if raw_json.startswith('```'):
    raw_json = raw_json.split('```')[1]
    raw_json = raw_json.removeprefix('json').strip()

structured = json.loads(raw_json)
print('🧾 Structured JSON output:\n')
print(json.dumps(structured, indent=2))

🧾 Structured JSON output:

{
  "seller": {
    "name": "Company Name",
    "address": "321 Avenue A, Portland, OR 12345",
    "phone": "(206) 555-1163",
    "fax": "(206) 555-1164",
    "email": "someone@websitegoeshere.com"
  },
  "invoice_metadata": {
    "date": "6/28/2024",
    "invoice_number": "1111",
    "po_number": "123456"
  },
  "bill_to": {
    "name": "Natasha Jones",
    "company": "Central Beauty",
    "address": "123 Main St., Manhattan, NY 98765",
    "phone": "(321) 555-1234",
    "discount": "10%"
  },
  "line_items": [
    {
      "quantity": "1",
      "description": "Item Number 1",
      "unit_price": "$2.00",
      "amount": "$2.00"
    },
    {
      "quantity": "1",
      "description": "Item Number 2",
      "unit_price": "$2.00",
      "amount": "$2.00"
    },
    {
      "quantity": "1",
      "description": "Item Number 3",
      "unit_price": "$2.00",
      "amount": "$2.00"
    }
  ],
  "totals": {
    "subtotal": "$6.00",
    "credit": "$1,000.00",
    

<div style="page-break-before: always;"></div>

<div style="border: 5px solid black; padding: 20px; border-radius: 6px;">

## Summary

Using the OpenAI API, I extracted the invoice text with one call and converted it into structured JSON with a second call. The pipeline worked well: faithful transcription, clean organization, and correct handling of the trickiest fields. The value of the exercise was less in proving the API can do this and more in seeing where it makes unannounced judgment calls, which is exactly where human review has to stay in the loop.

---

## Reflection

Splitting the work into two calls paid off because it lets me separate what the model read from what the model decided. The extraction step was essentially perfect. It transcribed every value verbatim, left placeholder fields like <Name> and <Company name> untouched, kept the empty rows, and preserved the balance due in its original parenthetical form. Anything I question below comes from the structuring step, not from a misread page.

The structuring step got the hard formatting right. The dollar amounts came back as clean currency strings such as `"$2.00"` and `"$1,000.00"`, the percentages stayed as written, and the balance due is `"-$994.20"`, which shows the model read the parentheses as a negative rather than as decoration. That point matters most because the `$1,000` credit against a `$6.00` subtotal means the customer is actually owed money, and the sign survived. With every formatted value stored as a string, the typing is also consistent, though it would need to be parsed before any math.

The more revealing results are the things the model decided on its own. Two real fields from the invoice at times when running the notebook, don't make it into the structured JSON call: the `"% discount 10%"` line and the note that items over `$100` qualify for an additional discount, when I originally wrote this reflection, neither appeared, but after running one final time before submission, the line about the discount qualifier, appears. Both appear in the raw transcription, so the model did not fail to read them; at times, it seemingly chooses not to carry them forward. Nothing flagged the omission. The JSON looks complete, and only a line-by-line check against the source reveals the gap. The model also made structural calls I did not specify, collapsing the address fields into single strings and merging the three footer lines into one note instead of a list. Those are reasonable readings, but they are readings, not the only correct answer.

All of this lines up with the math problem from earlier in the assignment. The model is fast, it reads well, and it organizes confidently, but it makes silent choices when the input is ambiguous, and it can quietly drop real information without saying so. For a financial document, especially, that is the whole argument for keeping a human in the loop. "Looks right" and "is right" are not the same thing, and the only way to tell them apart here was to read the JSON back against the invoice.

</div>